# LoRA v2 — Confidence-Based Inference Router

- Uses your best model (v2) with no retraining
- Runs primary inference with native resolution images (best at 0.768)
- Measures confidence margin — the gap between the top-1 and top-2 answer scores
- High-confidence predictions are kept as-is
- Low-confidence predictions get a second opinion: re-run with 224×224 images or average logits from both native and 224
- The idea: the model is already right on easy questions, so we only intervene on uncertain ones where a different image processing might flip the answer
- Zero training cost — purely an inference-time strategy

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.2 MB/s eta 0:00:00


In [2]:
import kagglehub
from pathlib import Path
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

100%|██████████| 358M/358M [00:23<00:00, 16.2MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


In [5]:
import os, json, random, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"
BATCH_SIZE_INFER = 4

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists(): DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 3109 | Val: 1048 | Test: 1008


In [7]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

## Cell 5: Load v2

In [8]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/lora_v2_best")
model.eval()
candidate_ids = get_candidate_token_ids(processor.tokenizer)
print("Loaded v2 checkpoint")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded v2 checkpoint


## Cell 6: Confidence Router

In [9]:
def prompt_full(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

def predict_with_scores(df_batch, img_fn, prompt_fn):
    """Returns preds AND confidence margins."""
    images = [img_fn(row) for _, row in df_batch.iterrows()]
    prompts = [prompt_fn(row) for _, row in df_batch.iterrows()]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]
    log_probs = torch.log_softmax(logits, dim=-1)
    preds, margins = [], []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = [log_probs[i, candidate_ids[CHOICE_LETTERS[ci]]].max().item()
                  for ci in range(len(row["choices"]))]
        sorted_scores = sorted(scores, reverse=True)
        margin = sorted_scores[0] - sorted_scores[1]  # gap between top 2
        preds.append(int(np.argmax(scores)))
        margins.append(margin)
    return preds, margins

# Image loaders
def img_native(row):
    return Image.open(DATA_ROOT / row["image_path"]).convert("RGB")

def img_224(row):
    return Image.open(DATA_ROOT / row["image_path"]).convert("RGB").resize((224,224), Image.BICUBIC)

# Run primary (native + full) on all samples
eval_df = val_df.sample(n=200, random_state=SEED).reset_index(drop=True)

print("Running primary inference (native + full prompt)...")
primary_preds, primary_margins = [], []
for start in range(0, len(eval_df), BATCH_SIZE_INFER):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    try:
        p, m = predict_with_scores(batch, img_native, prompt_full)
    except RuntimeError:
        torch.cuda.empty_cache()
        p, m = [], []
        for j in range(len(batch)):
            pp, mm = predict_with_scores(batch.iloc[j:j+1], img_native, prompt_full)
            p.extend(pp); m.extend(mm)
            torch.cuda.empty_cache()
    primary_preds.extend(p)
    primary_margins.extend(m)
    torch.cuda.empty_cache()

eval_df["primary_pred"] = primary_preds
eval_df["margin"] = primary_margins
eval_df["primary_correct"] = eval_df["primary_pred"] == eval_df["answer"]
primary_acc = eval_df["primary_correct"].mean()
print(f"Primary accuracy: {primary_acc:.4f}")

# Analyze confidence distribution
print(f"\nMargin stats:")
print(f"  Mean: {np.mean(primary_margins):.3f}")
print(f"  Median: {np.median(primary_margins):.3f}")
print(f"  Low-conf (<1.0): {sum(m < 1.0 for m in primary_margins)} samples")
print(f"  Low-conf (<0.5): {sum(m < 0.5 for m in primary_margins)} samples")

# Accuracy by confidence bucket
for threshold in [0.5, 1.0, 2.0, 3.0]:
    high = eval_df[eval_df["margin"] >= threshold]
    low = eval_df[eval_df["margin"] < threshold]
    if len(high) > 0 and len(low) > 0:
        print(f"  Margin >= {threshold}: {high['primary_correct'].mean():.3f} ({len(high)} samples) | "
              f"< {threshold}: {low['primary_correct'].mean():.3f} ({len(low)} samples)")

Running primary inference (native + full prompt)...
Primary accuracy: 0.6750

Margin stats:
  Mean: 2.794
  Median: 1.703
  Low-conf (<1.0): 68 samples
  Low-conf (<0.5): 39 samples
  Margin >= 0.5: 0.720 (161 samples) | < 0.5: 0.487 (39 samples)
  Margin >= 1.0: 0.758 (132 samples) | < 1.0: 0.515 (68 samples)
  Margin >= 2.0: 0.793 (92 samples) | < 2.0: 0.574 (108 samples)
  Margin >= 3.0: 0.845 (58 samples) | < 3.0: 0.606 (142 samples)


## Cell 7: Run fallback on low-confidence samples

In [10]:
# Try 224 + full prompt as fallback (different image processing might help)
THRESHOLD = 1.0  # Adjust based on cell 6 output

low_conf_idx = eval_df[eval_df["margin"] < THRESHOLD].index.tolist()
print(f"\nRe-running {len(low_conf_idx)} low-confidence samples with 224 + full prompt...")

fallback_preds = {}
for idx in tqdm(low_conf_idx):
    row = eval_df.iloc[idx:idx+1]
    p, m = predict_with_scores(row, img_224, prompt_full)
    fallback_preds[idx] = p[0]
    torch.cuda.empty_cache()

# Build routed predictions
eval_df["routed_pred"] = eval_df["primary_pred"].copy()
for idx, pred in fallback_preds.items():
    eval_df.loc[idx, "routed_pred"] = pred

eval_df["routed_correct"] = eval_df["routed_pred"] == eval_df["answer"]
routed_acc = eval_df["routed_correct"].mean()

print(f"\nPrimary (native):  {primary_acc:.4f}")
print(f"Routed:            {routed_acc:.4f}")
print(f"Change:            {routed_acc - primary_acc:+.4f}")

# Also try: average logits from both instead of picking one
print("\n--- Trying logit averaging on low-conf samples ---")

def predict_averaged(df_row):
    """Average log-probs from native and 224 for one sample."""
    row = df_row.iloc[0] if len(df_row) == 1 else df_row

    img_n = img_native(row)
    img_2 = img_224(row)
    prompt = prompt_full(row)

    scores_list = []
    for img in [img_n, img_2]:
        inputs = processor(text=[prompt], images=[img], return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
        with torch.inference_mode():
            logits = model(**inputs).logits[:, -1, :]
        log_probs = torch.log_softmax(logits, dim=-1)
        scores = [log_probs[0, candidate_ids[CHOICE_LETTERS[ci]]].max().item()
                  for ci in range(len(row["choices"]))]
        scores_list.append(scores)

    # Average scores
    avg_scores = [np.mean([s[ci] for s in scores_list]) for ci in range(len(scores_list[0]))]
    return int(np.argmax(avg_scores))

avg_preds = {}
for idx in tqdm(low_conf_idx, desc="Averaging"):
    row = eval_df.iloc[idx]
    avg_preds[idx] = predict_averaged(pd.DataFrame([row]))
    torch.cuda.empty_cache()

eval_df["avg_pred"] = eval_df["primary_pred"].copy()
for idx, pred in avg_preds.items():
    eval_df.loc[idx, "avg_pred"] = pred

eval_df["avg_correct"] = eval_df["avg_pred"] == eval_df["answer"]
avg_acc = eval_df["avg_correct"].mean()

print(f"\nPrimary (native):  {primary_acc:.4f}")
print(f"Routed (fallback): {routed_acc:.4f}")
print(f"Averaged (logits): {avg_acc:.4f}")

best_method = max([("primary", primary_acc), ("routed", routed_acc), ("averaged", avg_acc)], key=lambda x: x[1])
print(f"\nBest: {best_method[0]} ({best_method[1]:.4f})")


Re-running 68 low-confidence samples with 224 + full prompt...


  0%|          | 0/68 [00:00<?, ?it/s]


Primary (native):  0.6750
Routed:            0.6800
Change:            +0.0050

--- Trying logit averaging on low-conf samples ---


Averaging:   0%|          | 0/68 [00:00<?, ?it/s]


Primary (native):  0.6750
Routed (fallback): 0.6800
Averaged (logits): 0.6700

Best: routed (0.6800)


## Cell 8: Submit best method

In [11]:
best = best_method[0]
print(f"Submitting: {best}")

all_preds = []
all_margins = []

# First pass: primary (native)
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Primary"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    try:
        p, m = predict_with_scores(batch, img_native, prompt_full)
    except RuntimeError:
        torch.cuda.empty_cache()
        p, m = [], []
        for j in range(len(batch)):
            pp, mm = predict_with_scores(batch.iloc[j:j+1], img_native, prompt_full)
            p.extend(pp); m.extend(mm)
            torch.cuda.empty_cache()
    all_preds.extend(p)
    all_margins.extend(m)
    torch.cuda.empty_cache()

if best == "primary":
    final_preds = all_preds
else:
    # Re-run low confidence samples
    final_preds = list(all_preds)
    low_conf = [i for i, m in enumerate(all_margins) if m < THRESHOLD]
    print(f"Re-running {len(low_conf)} low-confidence test samples...")

    for idx in tqdm(low_conf, desc="Fallback"):
        row = test_df.iloc[idx:idx+1]
        if best == "routed":
            p, _ = predict_with_scores(row, img_224, prompt_full)
            final_preds[idx] = p[0]
        else:  # averaged
            final_preds[idx] = predict_averaged(row)
        torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": final_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_router.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")

Submitting: routed


Primary:   0%|          | 0/252 [00:00<?, ?it/s]

Re-running 263 low-confidence test samples...


Fallback:   0%|          | 0/263 [00:00<?, ?it/s]

Saved (1008 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>